In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path.cwd().parent / 'data' / 'interim'
ONBOARDING_WINDOW = 90

eligible         = pd.read_csv(DATA / 'eligible_with_labels.csv')
new_clients      = pd.read_csv(DATA / 'new_clients.csv')
tx_with_reg      = pd.read_csv(DATA / 'tx_with_reg.csv')
messages         = pd.read_csv(DATA / 'messages.csv')
message_templates= pd.read_csv(DATA / 'message_templates.csv')
conversions      = pd.read_csv(DATA / 'conversions.csv')
business_units   = pd.read_csv(DATA / 'business_units.csv')

for df, col in [
    (new_clients, 'registration_date'),
    (tx_with_reg, 'date'),
    (tx_with_reg, 'registration_date'),
    (messages,    'send_date'),
    (conversions, 'date'),
]:
    df[col] = pd.to_datetime(df[col], utc=True)

print(f'eligible: {len(eligible):,} | tx_with_reg: {len(tx_with_reg):,}')


In [ ]:
# First transaction day per client (reference point for all features)
cols_to_drop = [c for c in tx_with_reg.columns
                if c in ['day_first_tx_x','day_first_tx_y','day_first_tx','days_since_first_tx']]
tx_with_reg = tx_with_reg.drop(columns=cols_to_drop)

first_tx = (
    tx_with_reg.groupby('client_id')['days_since_reg'].min()
    .reset_index().rename(columns={'days_since_reg': 'day_first_tx'})
)
tx_with_reg = tx_with_reg.merge(first_tx, on='client_id', how='left')
tx_with_reg['days_since_first_tx'] = tx_with_reg['days_since_reg'] - tx_with_reg['day_first_tx']

# Transactions within 90-day onboarding window
tx_onboarding = tx_with_reg[
    (tx_with_reg['days_since_first_tx'] >= 0) &
    (tx_with_reg['days_since_first_tx'] <= ONBOARDING_WINDOW) &
    (tx_with_reg['client_id'].isin(eligible['client_id']))
].copy()
tx_onboarding = tx_onboarding.merge(
    business_units[['id', 'location', 'category']],
    left_on='business_unit_id', right_on='id', how='left'
)
print(f'tx_onboarding: {len(tx_onboarding):,} rows | {tx_onboarding["client_id"].nunique():,} clients')


In [ ]:
# Median inter-visit gap per format (from first inter-visit gap, all eligible clients)
format_median_gap = (
    tx_with_reg[tx_with_reg['client_id'].isin(eligible['client_id'])]
    .merge(business_units[['id','location']], left_on='business_unit_id', right_on='id', how='left')
    .assign(tx_date=lambda x: x['days_since_first_tx'].round(0))
    .drop_duplicates(subset=['client_id','tx_date'])
    .sort_values(['client_id','tx_date'])
    .assign(prev_day=lambda x: x.groupby('client_id')['tx_date'].shift(1))
    .assign(gap=lambda x: x['tx_date'] - x['prev_day'])
    .dropna(subset=['gap'])
    .groupby('client_id').first().reset_index()
    .merge(
        tx_onboarding.sort_values('days_since_first_tx')
        .groupby('client_id')['location'].first().reset_index()
        .rename(columns={'location':'first_location'}),
        on='client_id', how='left'
    )
    .groupby('first_location')['gap'].median().reset_index()
    .rename(columns={'gap':'median_gap'})
)
global_median = format_median_gap['median_gap'].median()
print('Format median gap (days):'); print(format_median_gap.sort_values('median_gap').to_string(index=False))
print(f'Global median (fallback): {global_median:.0f} days')


In [ ]:
# RFM features
tx_count = (tx_onboarding.groupby('client_id')['transaction_id'].count()
            .reset_index().rename(columns={'transaction_id':'tx_count'}))

unique_days = (
    tx_onboarding.assign(tx_date=lambda x: x['days_since_first_tx'].round(0))
    .drop_duplicates(subset=['client_id','tx_date'])
    .groupby('client_id')['tx_date'].count()
    .reset_index().rename(columns={'tx_date':'unique_active_days'})
)

recency = (tx_onboarding.groupby('client_id')['days_since_first_tx'].max()
           .reset_index().rename(columns={'days_since_first_tx':'last_tx_day'}))
recency['recency'] = ONBOARDING_WINDOW - recency['last_tx_day']
recency = recency[['client_id','recency']]

monetary = (tx_onboarding.groupby('client_id').agg(
    total_spent=('payed_amount','sum'), avg_check=('payed_amount','mean'),
    bonuses_accum=('bonusses_accum','sum'), bonuses_used=('bonusses_used','sum')
).reset_index())

tx_sorted = (
    tx_onboarding.assign(tx_date=lambda x: x['days_since_first_tx'].round(0))
    .drop_duplicates(subset=['client_id','tx_date']).sort_values(['client_id','tx_date'])
)
tx_sorted['prev_day'] = tx_sorted.groupby('client_id')['tx_date'].shift(1)
tx_sorted['gap'] = tx_sorted['tx_date'] - tx_sorted['prev_day']
avg_gap = (tx_sorted.groupby('client_id')['gap'].mean()
           .reset_index().rename(columns={'gap':'avg_days_between_tx'}))
avg_gap['avg_days_between_tx'] = avg_gap['avg_days_between_tx'].fillna(0)

# Activity trend: 2nd half minus 1st half (45-day split)
first_half  = (tx_onboarding[tx_onboarding['days_since_first_tx'] <= 45]
               .groupby('client_id')['transaction_id'].count()
               .reset_index().rename(columns={'transaction_id':'tx_first_half'}))
second_half = (tx_onboarding[tx_onboarding['days_since_first_tx'] > 45]
               .groupby('client_id')['transaction_id'].count()
               .reset_index().rename(columns={'transaction_id':'tx_second_half'}))
trend = first_half.merge(second_half, on='client_id', how='outer').fillna(0)
trend['activity_trend'] = trend['tx_second_half'] - trend['tx_first_half']
trend = trend[['client_id','activity_trend']]

first_location = (
    tx_onboarding.sort_values('days_since_first_tx')
    .groupby('client_id')[['location','category']].first().reset_index()
    .rename(columns={'location':'first_location','category':'first_category'})
)
first_location = first_location.merge(format_median_gap, on='first_location', how='left')
first_location['median_gap'] = first_location['median_gap'].fillna(global_median)

unique_locations = (tx_onboarding.groupby('client_id')['location'].nunique()
                    .reset_index().rename(columns={'location':'unique_locations'}))
online_share = (tx_onboarding.groupby('client_id')
                .apply(lambda x: (x['category'] == 'Online purchases').mean())
                .reset_index().rename(columns={0:'online_tx_share'}))

# Format-normalised features
recency_norm = recency.merge(first_location[['client_id','median_gap']], on='client_id', how='left')
recency_norm['recency_normalized'] = recency_norm['recency'] / recency_norm['median_gap']
recency_norm = recency_norm[['client_id','recency_normalized']]

visit_fulfill = unique_days.merge(first_location[['client_id','median_gap']], on='client_id', how='left')
visit_fulfill['expected_visits'] = ONBOARDING_WINDOW / visit_fulfill['median_gap']
visit_fulfill['visit_fulfillment'] = visit_fulfill['unique_active_days'] / visit_fulfill['expected_visits']
visit_fulfill = visit_fulfill[['client_id','visit_fulfillment']]

print('Transaction features done')


In [ ]:
# Message segment groups (3 groups for 90-day model)
segment_groups = {
    'Thank-you / post-visit': 'auto_trigger',
    'Quest / cross-venue':    'auto_trigger',
    'Loyalty: double points':       'retention_offer',
    'Loyalty: points spend/expiry': 'retention_offer',
    'Coupon / discount':            'retention_offer',
}
message_templates['segment_group'] = message_templates['segment'].map(segment_groups).fillna('other')

bridge = new_clients[['client_id','loyalty_client_id']].copy()

messages_new = (
    messages.dropna(subset=['client_id'])
    .merge(bridge, left_on='client_id', right_on='loyalty_client_id', how='inner')
    .rename(columns={'client_id_x':'loyalty_client_id_msg','client_id_y':'client_id'})
    .merge(new_clients[['client_id','registration_date']], on='client_id', how='left')
    .merge(first_tx, on='client_id', how='left')
)
messages_new['days_since_reg'] = (
    messages_new['send_date'] - messages_new['registration_date']
).dt.total_seconds() / 86400
messages_new['days_since_first_tx'] = messages_new['days_since_reg'] - messages_new['day_first_tx']

msg_onboarding = messages_new[
    (messages_new['days_since_first_tx'] >= 0) &
    (messages_new['days_since_first_tx'] <= ONBOARDING_WINDOW) &
    (messages_new['client_id'].isin(eligible['client_id']))
].copy()
msg_onboarding = msg_onboarding.merge(
    message_templates[['message_template_id','segment_group']], on='message_template_id', how='left'
)

msg_features = (
    msg_onboarding.groupby('client_id')
    .agg(_n=('message_id','count'), _opened=('status', lambda x: (x=='Opened').sum()))
    .reset_index()
)
msg_features['open_rate'] = msg_features['_opened'] / msg_features['_n']
msg_features = msg_features[['client_id','open_rate']]

# msg_seg_retention_offer: 90-day model only (not in prod/30-day model)
retention_offer_counts = (
    msg_onboarding[msg_onboarding['segment_group'] == 'retention_offer']
    .groupby('client_id').size().reset_index(name='msg_seg_retention_offer')
)
print(f'msg_onboarding: {len(msg_onboarding):,} | clients: {msg_onboarding["client_id"].nunique():,}')


In [ ]:
# Conversion features
conversions_new = (
    conversions
    .merge(bridge, left_on='client_id', right_on='loyalty_client_id', how='inner')
    .rename(columns={'client_id_x':'loyalty_client_id_conv','client_id_y':'client_id'})
    .merge(new_clients[['client_id','registration_date']], on='client_id', how='left')
    .merge(first_tx, on='client_id', how='left')
)
conversions_new['days_since_reg'] = (
    conversions_new['date'] - conversions_new['registration_date']
).dt.total_seconds() / 86400
conversions_new['days_since_first_tx'] = conversions_new['days_since_reg'] - conversions_new['day_first_tx']

conv_onboarding = conversions_new[
    (conversions_new['days_since_first_tx'] >= 0) &
    (conversions_new['days_since_first_tx'] <= ONBOARDING_WINDOW) &
    (conversions_new['client_id'].isin(eligible['client_id']))
].copy()
conv_features = (
    conv_onboarding.groupby('client_id').agg(conv_count=('conversion_id','count')).reset_index()
)
print(f'conv_onboarding: {len(conv_onboarding):,} | clients: {conv_onboarding["client_id"].nunique():,}')


In [ ]:
# Registration profile features
profile = new_clients[new_clients['client_id'].isin(eligible['client_id'])][['client_id','registration_date']].copy()
profile['registration_date'] = pd.to_datetime(profile['registration_date']).dt.tz_localize(None)
profile['reg_day_of_week'] = profile['registration_date'].dt.dayofweek
profile['reg_month']       = profile['registration_date'].dt.month
profile = profile[['client_id','reg_day_of_week','reg_month']]
print('Profile features done')


In [ ]:
# Assemble feature matrix
features = eligible[['client_id','churned']].copy()
for df in [tx_count, unique_days, recency, monetary, avg_gap, trend,
           first_location, unique_locations, online_share, recency_norm, visit_fulfill]:
    features = features.merge(df, on='client_id', how='left')
features = features.merge(msg_features, on='client_id', how='left')
features = features.merge(retention_offer_counts, on='client_id', how='left')
features = features.merge(conv_features, on='client_id', how='left')
features = features.merge(profile, on='client_id', how='left')

# Fill NaN
features[['avg_check','total_spent','msg_seg_retention_offer','conv_count',
          'open_rate','bonuses_accum','bonuses_used','unique_locations','online_tx_share']] = \
    features[['avg_check','total_spent','msg_seg_retention_offer','conv_count',
              'open_rate','bonuses_accum','bonuses_used','unique_locations','online_tx_share']].fillna(0)
features['first_location'] = features['first_location'].fillna('Unknown')
features['first_category'] = features['first_category'].fillna('Unknown')
features = features.drop(columns=['median_gap','day_first_tx','last_tx_day'], errors='ignore')

print(f'features: {features.shape}  NaN remaining: {features.isna().sum().sum()}')
print(f'Churn rate: {features["churned"].mean()*100:.1f}%')
features.to_csv(DATA / 'features.csv', index=False)
msg_onboarding.to_csv(DATA / 'msg_onboarding.csv', index=False)
conv_onboarding.to_csv(DATA / 'conv_onboarding.csv', index=False)
print('Saved: features.csv, msg_onboarding.csv, conv_onboarding.csv')
